# Import Required Packages

In [21]:
# Basic Packages
import os
import sys
import warnings
import pickle
warnings.filterwarnings("ignore")
from typing import List


# Standard ML Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin


# Tensorflow packages
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, 
                                    Dense)
from tensorflow.keras.optimizers import (Adam,
                                         AdamW)
from tensorflow.keras.losses import (CategoricalCrossentropy, 
                                     SparseCategoricalCrossentropy)

from tensorflow.keras.callbacks import (EarlyStopping, 
                                        ModelCheckpoint,
                                        TensorBoard)

# Load the Training Samples

In [16]:
with open("../artifacts/trainingData/training_ready_samples.pkl", "rb") as f:
    training_samples = pickle.load(f)
    print("Done!! Loading the Training Samples")

Done!! Loading the Training Samples


In [ ]:
training_generator = tf.data.Dataset.from_tensor_slices(training_samples).batch(64)


<_BatchDataset element_spec=(TensorSpec(shape=(None, 4), dtype=tf.int64, name=None), TensorSpec(shape=(None, 1), dtype=tf.int64, name=None))>

# Keras Tuner Neural Network

#### Step 1: Neural Network Architecture

In [42]:
class CustomEmbeddingsModel(Model):

    def __init__(self, hp, num_classes):
        super(CustomEmbeddingsModel, self).__init__()

        # Tunable Parameter for layer 1
        dense_units_1 = hp.Int("units", min_value = 256, max_value = 2058, step = 128)

        # Dense layers
        self.dense_layer_1 = Dense(units = dense_units_1, 
                                 activation = "leaky_relu",
                                 )

        # Dense layers
        self.dense_layer_2 = Dense(units = 128, 
                                 activation = "leaky_relu")
        self.dense_layer_3 = Dense(units = 128, 
                                   activation = "leaky_relu")
        self.output_layer = Dense(units = num_classes, 
                                   activation = "softmax") 


    def call(self, inputs):

        # Dense layer 
        x = self.dense_layer_1(inputs)
        x = self.dense_layer_2(x)
        x = self.dense_layer_3(x)

        # Output layer
        out = self.output_layer(x)

        return out       


#### Step 2 : Keras Tuner Hyperband Model

In [43]:
def keras_tuner_hyperband_model(hp):

    model = CustomEmbeddingsModel(hp, num_classes=2006)

    # Compile the model 
    hp_optimizers = hp.Choice("optimizers", values = ["Adam", "AdamW"])
    model.compile(optimizer = hp_optimizers, loss = SparseCategoricalCrossentropy(),
                  metrics = ['accuracy','f1_score'])
    
    return model

#### Step 3: Instantiate the tuner and perform hypertuning

In [44]:
tuner = kt.Hyperband(keras_tuner_hyperband_model, 
                     max_epochs = 3, 
                     directory = "../model/",
                     project_name = "cbow_tuner")


Reloading Tuner from ../model/cbow_tuner/tuner0.json


#### Step 4: Early Stopping

In [45]:
early_stopping_callback = EarlyStopping(
    verbose = 0, 
    mode = "min",
    patience = 10
)

callbacks = [early_stopping_callback]

# Step 5: Tuner Search

In [46]:
tuner.search(training_generator, 
             epochs = 3, 
             callbacks = callbacks)


# Best Hyperparametersm
best_hps = tuner.get_best_hyperparameters()[0]

print(f"""
The hyperparameter search is complete. The optimal number of units in the first LSTM layer is {best_hps.get("units")} and the best choice of optimizer is
{best_hps.get("optimizers")}
""")


Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
1301              |320               |units
Adam              |AdamW             |optimizers
1                 |1                 |tuner/epochs
0                 |0                 |tuner/initial_epoch
1                 |1                 |tuner/bracket
0                 |0                 |tuner/round



Traceback (most recent call last):
  File "/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
  File "/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/keras_tuner/src/tuners/hyperband.py", line 427, in run_trial
    return super().run_trial(trial, *fit_args, **fit_kwargs)
  File "/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/pyt

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
  File "/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/keras_tuner/src/tuners/hyperband.py", line 427, in run_trial
    return super().run_trial(trial, *fit_args, **fit_kwargs)
  File "/Users/kavisanthoshkumar/Library/CloudStorage/On